# 🎬 EducationVdo - Colab Runner

Run **EducationVdo** with full GPU acceleration on Google Colab.
Supports face swapping, face enhancement, expression restoration, and **image-to-video AI animation (LivePortrait)**.

In [ ]:
# ── Check GPU Availability ───────────────────────────────
!nvidia-smi

In [ ]:
# ── Step 1: Clone Repository ─────────────────────────────
from getpass import getpass
import os

print("If your repository is private, enter your GitHub Personal Access Token.")
print("Generate one at: https://github.com/settings/tokens")
print("Leave empty if the repository is public.")
print()

token = getpass("GitHub Token (optional): ")

repo_url = "https://github.com/hasnatvip/educatonvdo.git"
if token.strip():
    repo_url = repo_url.replace("https://", f"https://{token}@")

# Remove existing directory if it exists
!rm -rf educatonvdo

# Clone the repository
!git clone {repo_url}

# Change to the project directory
%cd educatonvdo

In [ ]:
# ── Step 2: Install Dependencies & Setup CUDA 12 ────────
import os, sys
for candidate in ['/content/educatonvdo', '/content/workspace', os.getcwd()]:
    if os.path.exists(os.path.join(candidate, 'educationvdo.py')):
        os.chdir(candidate)
        break
print(f'Working directory: {os.getcwd()}')
print('Installing dependencies and CUDA 12 runtime libraries...')

# Ensure clean onnxruntime & cuDNN environment
!pip uninstall -q -y onnxruntime onnxruntime-directml onnxruntime-openvino nvidia-cudnn-cu12 2>/dev/null || true

# Install project requirements
!pip install -q -r requirements.txt

# Install NVIDIA CUDA 12 & stable cuDNN 9 libraries (pinned to 9.7.1.26 to prevent experimental 9.27.0 which breaks ConvFwd)
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.7.1.26"

# Configure library search paths for dynamic linker
import glob, site
nvidia_lib_dirs = []
for p in site.getsitepackages() + sys.path:
    if os.path.isdir(p):
        for lib_dir in glob.glob(os.path.join(p, 'nvidia', '*', 'lib')):
            if os.path.isdir(lib_dir) and lib_dir not in nvidia_lib_dirs:
                nvidia_lib_dirs.append(lib_dir)

cuda_system_dirs = ['/usr/local/cuda/lib64', '/usr/local/cuda-12/lib64']
all_lib_dirs = [d for d in nvidia_lib_dirs + cuda_system_dirs if os.path.isdir(d)]
existing_ld = os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = ':'.join(all_lib_dirs + ([existing_ld] if existing_ld else []))

# Preload CUDA & cuDNN 9 sub-libraries into memory
import ctypes
priority_prefixes = [
    'libcuda.so', 'libculibos.so', 'libcurand.so', 'libcufft.so',
    'libcublasLt.so', 'libcublas.so',
    'libcudnn_ops.so', 'libcudnn_cnn.so', 'libcudnn_adv.so', 'libcudnn_graph.so',
    'libcudnn_engines_runtime_compiled.so', 'libcudnn_engines_precompiled.so',
    'libcudnn_heuristic.so', 'libcudnn.so'
]
loaded = set()
for prefix in priority_prefixes:
    for directory in all_lib_dirs:
        for file_path in sorted(glob.glob(os.path.join(directory, prefix + '*'))):
            real = os.path.realpath(file_path)
            if real not in loaded:
                try:
                    ctypes.CDLL(real, mode=ctypes.RTLD_GLOBAL)
                    loaded.add(real)
                except Exception:
                    pass

# Verify installation & CUDA Provider
import onnxruntime
if hasattr(onnxruntime, 'preload_dlls'):
    try:
        onnxruntime.preload_dlls()
    except Exception:
        pass

providers = onnxruntime.get_available_providers()
print()
print(f'✓ ONNX Runtime: {onnxruntime.__version__}')
print(f'✓ Available Execution Providers: {providers}')

if 'CUDAExecutionProvider' in providers:
    print('🚀 CUDA Acceleration is READY!')
else:
    print('⚠️ Notice: Running with CPU (CUDA provider not initialized)')

print()
print('✓ Setup completed successfully!')

In [ ]:
# ── Step 3: Launch EducationVdo UI ───────────────────────
import os, sys, subprocess

# Ensure we are in the project directory
for candidate in ['/content/educatonvdo', '/content/workspace', os.getcwd()]:
    if os.path.exists(os.path.join(candidate, 'educationvdo.py')):
        os.chdir(candidate)
        break
print(f'[INFO] Working directory: {os.getcwd()}')

import onnxruntime
if hasattr(onnxruntime, 'preload_dlls'):
    try:
        onnxruntime.preload_dlls()
    except Exception:
        pass

# Ensure robust cuDNN convolution search mode and unbuffered live streaming
os.environ['EDUCATIONVDO_CUDNN_CONV_ALGO_SEARCH'] = 'DEFAULT'
os.environ['PYTHONUNBUFFERED'] = '1'

provider = 'cuda' if 'CUDAExecutionProvider' in onnxruntime.get_available_providers() else 'cpu'
print(f'🚀 Launching EducationVdo with {provider.upper()} execution provider...')
print('⏳ Please wait 20-30 seconds for the Gradio public URL to appear below...')
print('━' * 60)
sys.stdout.flush()

cmd = [
    sys.executable, '-u', 'educationvdo.py', 'run',
    '--execution-providers', provider
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ
)

for line in proc.stdout:
    print(line, end='', flush=True)

rc = proc.wait()
if rc != 0:
    print(f'
[ERROR] EducationVdo process exited with code {rc}')
